# Intelligent Plant Disease Detection System (IPDDS)
## Targeted Model Training: Potato & Tomato Leaf Diseases
This notebook trains the shared IPDDS models using the Potato and Tomato classes from both PlantVillage sources: `moazeldsokyx/plantvillage` and `emmarex/plantdisease`. Exact duplicate files are removed before the stratified train/validation split.

In [ ]:
import os
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

sys.path.insert(0, os.path.abspath(".."))
from train import (
    DISEASE_METADATA,
    POTATO_TOMATO_CLASSES,
    resolve_dataset_paths,
    resolve_additional_dataset_paths,
    create_stratified_splits,
    create_dataset
)

print(f"TensorFlow Version: {tf.__version__}")

### 1. Dataset Path & Target Classes
Both PlantVillage sources are loaded. Only the 13 Potato and Tomato disease classes supported by IPDDS are used; other crop classes are excluded.

In [ ]:
train_path, val_path = resolve_dataset_paths()
supplemental_sources = resolve_additional_dataset_paths()
train_paths = [train_path] + [source[0] for source in supplemental_sources]
val_paths = ([val_path] if val_path else []) + [
    source[1] for source in supplemental_sources if source[1]
]

print(f"Primary dataset directory: {train_path}")
print(f"Supplemental dataset directories: {[source[0] for source in supplemental_sources]}")
print(f"Selected IPDDS classes ({len(POTATO_TOMATO_CLASSES)}): {POTATO_TOMATO_CLASSES}")

### 2. Loading Filtered Image Datasets
Exact duplicate images across the two sources are removed, then each disease class is split separately into 80% training and 20% validation data.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
RANDOM_SEED = 42
train_examples, val_examples, train_counts, val_counts, duplicate_count = (
    create_stratified_splits(train_paths, val_paths or None, seed=RANDOM_SEED)
)
train_ds = create_dataset(train_examples, BATCH_SIZE, IMG_SIZE, shuffle=True, seed=RANDOM_SEED)
val_ds = create_dataset(val_examples, BATCH_SIZE, IMG_SIZE, shuffle=False, seed=RANDOM_SEED)

print(f"Exact duplicate images removed: {duplicate_count}")
print(f"Training images: {len(train_examples)} · Validation images: {len(val_examples)}")
print(f"Classes loaded ({len(POTATO_TOMATO_CLASSES)} total): {POTATO_TOMATO_CLASSES}")

### 3. Model Architecture with Data Augmentation

In [ ]:
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.Rescaling(1. / 255)(x)

x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(POTATO_TOMATO_CLASSES), activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="IPDDS_CNN")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

### 4. Training & Saving Model Weights

In [ ]:
os.makedirs("../model", exist_ok=True)
metadata_key_by_class = {
    "Tomato_Bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_Early_blight": "Tomato___Early_blight",
    "Tomato_Late_blight": "Tomato___Late_blight",
    "Tomato_Leaf_Mold": "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato__Target_Spot": "Tomato___Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato__Tomato_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_healthy": "Tomato___healthy"
}
class_metadata = {
    class_name: DISEASE_METADATA[metadata_key_by_class.get(class_name, class_name)]
    for class_name in POTATO_TOMATO_CLASSES
}
with open("../model/class_indices.json", "w", encoding="utf-8") as metadata_file:
    json.dump({
        "class_names": POTATO_TOMATO_CLASSES,
        "classes": class_metadata,
        "image_size": list(IMG_SIZE)
    }, metadata_file, indent=2)

checkpoint_cb = callbacks.ModelCheckpoint(
    "../model/disease_model_checkpoint.keras",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max"
)
early_stop_cb = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_cb, early_stop_cb]
)

model.save("../model/disease_model.keras")
model.save("../model/disease_model.h5")
print("Disease model and accurate class metadata saved to ../model/")

### 5. Train a Separate Crop-Type Classifier
A binary classifier is trained on the disease model's learned image features to distinguish Potato from Tomato independently of disease.

In [ ]:
feature_extractor = models.Model(
    inputs=model.input,
    outputs=model.get_layer("dense").output,
    name="IPDDS_Feature_Extractor"
)
crop_labels_by_class = tf.constant(
    [0 if class_name.startswith("Potato") else 1 for class_name in POTATO_TOMATO_CLASSES],
    dtype=tf.int32
)

def extract_crop_features(dataset):
    feature_batches = []
    crop_label_batches = []
    for images, disease_labels in dataset:
        feature_batches.append(feature_extractor(images, training=False).numpy())
        crop_label_batches.append(tf.gather(crop_labels_by_class, disease_labels).numpy())
    return np.concatenate(feature_batches), np.concatenate(crop_label_batches)

train_features, train_crop_labels = extract_crop_features(train_ds)
val_features, val_crop_labels = extract_crop_features(val_ds)
crop_counts = np.bincount(train_crop_labels, minlength=2)
crop_class_weights = {
    index: len(train_crop_labels) / (2 * count)
    for index, count in enumerate(crop_counts)
}

crop_head = models.Sequential([
    layers.Input(shape=(feature_extractor.output_shape[-1],)),
    layers.Dense(2, activation="softmax")
], name="IPDDS_Crop_Head")
crop_head.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
crop_history = crop_head.fit(
    train_features,
    train_crop_labels,
    validation_data=(val_features, val_crop_labels),
    class_weight=crop_class_weights,
    epochs=30,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)]
)

crop_model = models.Model(
    inputs=model.input,
    outputs=crop_head(feature_extractor.output),
    name="IPDDS_Crop_Classifier"
)
crop_model.save("../model/crop_type_model.keras")
print("Crop classifier saved to ../model/crop_type_model.keras (index 0: Potato, index 1: Tomato)")

disease_metrics = model.evaluate(val_ds, verbose=0, return_dict=True)
crop_metrics = crop_head.evaluate(val_features, val_crop_labels, verbose=0, return_dict=True)
training_metrics = {
    "validation_split": "stratified 80/20 per class after exact-file deduplication",
    "training_images": len(train_examples),
    "validation_images": len(val_examples),
    "duplicate_images_removed": duplicate_count,
    "training_datasets": ["moazeldsokyx/plantvillage", "emmarex/plantdisease"],
    "disease_accuracy": float(disease_metrics["accuracy"]),
    "crop_accuracy": float(crop_metrics["accuracy"]),
    "crop_classes": ["Potato", "Tomato"]
}
with open("../model/training_metrics.json", "w", encoding="utf-8") as metrics_file:
    json.dump(training_metrics, metrics_file, indent=2)

android_assets = "../android/app/src/main/assets"
os.makedirs(android_assets, exist_ok=True)
converted_models = {
    "disease_model.tflite": tf.lite.TFLiteConverter.from_keras_model(model).convert(),
    "crop_type_model.tflite": tf.lite.TFLiteConverter.from_keras_model(crop_model).convert()
}
for model_name, model_bytes in converted_models.items():
    with open(os.path.join(android_assets, model_name), "wb") as model_file:
        model_file.write(model_bytes)
with open(os.path.join(android_assets, "class_indices.json"), "w", encoding="utf-8") as metadata_file:
    json.dump({
        "class_names": POTATO_TOMATO_CLASSES,
        "classes": class_metadata,
        "image_size": list(IMG_SIZE)
    }, metadata_file, indent=2)
with open(os.path.join(android_assets, "training_metrics.json"), "w", encoding="utf-8") as metrics_file:
    json.dump(training_metrics, metrics_file, indent=2)
print(f"Disease validation accuracy: {training_metrics['disease_accuracy']:.4f}")
print(f"Crop validation accuracy: {training_metrics['crop_accuracy']:.4f}")
print("Keras and Android TensorFlow Lite models are ready for both apps.")